BLUEPRINT

I conducted exploratory data analysis (EDA) using Python to investigate effective feature engineering strategies and model architectures for TVT prediction.(https://www.kaggle.com/code/torrreeesss/rogii-pred-eda)
Based on the insights obtained from the EDA, I designed a prediction pipeline in which the target variable is defined as the residual TVT, namely the difference from the last known TVT value in the observed interval. The model integrates various features, including formation surfaces, ANCC interpolation, GR sequence similarity, trajectory information, and estimates obtained from particle filtering. I then implemented a GBDT stacking framework using commonly used tree-based models: LightGBM, CatBoost, and XGBoost. Finally, post-processing was applied to smooth the predicted values.
(https://www.kaggle.com/code/torrreeesss/tvt-pred-gbdt-stacking)
More specifically, LightGBM, CatBoost, and XGBoost were trained using GroupKFold cross-validation grouped by well. In addition to the out-of-fold predictions from each model, direct prediction signals from ANCC, beam search, particle filtering, and self-correlation-based methods were also incorporated as candidate predictors. These candidates were then combined using positive Ridge stacking and a hill-climbing ensemble to generate the final prediction. In the final stage, I applied post-processing based on blending with physically motivated signals, MD-distance-based adjustment, and well-wise Savitzky–Golay smoothing.

Pythonを用いてEDAを実施し、TVT予測に有効な特徴量設計とモデル構成を検討してみました。(https://www.kaggle.com/code/torrreeesss/rogii-pred-eda)
そこで得たデータから、既知区間の最後のTVTからの差分を目的変数として学習し、formation surface、ANCC補間、GR系列の類似度、trajectory情報、particle filterによる推定値などを特徴量として統合し、LightGBM、CatBoost、XGBoostというよくある決定木モデルを用いたGBDT stackingを行い、最終的にpost-processingによって予測値を平滑化する構成を実施しました。
（https://www.kaggle.com/code/torrreeesss/tvt-pred-gbdt-stacking）
次に、そのデータを用いて、well単位のGroupKFoldでLightGBM・CatBoost・XGBoostを学習しています。
各モデルのOOF予測に加えて、ANCC・beam・particle filter・self-correlation系の直接推定値も候補として統合し、positive Ridge stackingとhill-climbing ensembleで最終予測を作成してます。最後に、物理的な信号とのブレンドやMD距離に基づくpost-processing、well単位のSavgol平滑化を行いました。

In [ ]:
from __future__ import annotations

import copy
import gc
import multiprocessing
import os
import random
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostRegressor, Pool
from joblib import Parallel, delayed
from numba import njit
from scipy.signal import savgol_filter
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold

try:
    from sklearn.metrics import root_mean_squared_error
except Exception: 
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings("ignore")
!rm -rf /kaggle/working/data_cache
!rm -rf /kaggle/working/models


def find_dataset_path() -> Path:
    candidates = [
        Path("/kaggle/input/rogii-wellbore-geology-prediction"),
        Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction"),
    ]
    for p in candidates:
        if (p / "train").exists() and (p / "test").exists() and (p / "sample_submission.csv").exists():
            return p

    root = Path("/kaggle/input")
    if root.exists():
        for sample in root.glob("**/sample_submission.csv"):
            p = sample.parent
            if (p / "train").exists() and (p / "test").exists():
                return p
    return candidates[0]


class CFG:
    dataset_path = find_dataset_path()
    artifacts_path = Path("/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts")
    out_dir = Path("/kaggle/working")
    model_dir = out_dir / "models"
    data_cache_dir = out_dir / "data_cache"

    seed = 42
    n_splits = 5
    n_jobs = min(4, multiprocessing.cpu_count())
    use_artifact_features = False
    force_rebuild_features = True
    pp_trials = 150
    pp_n_jobs = 1
    early_stopping_rounds = 80


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def rmse(y_true, y_pred) -> float:
    return float(root_mean_squared_error(y_true, y_pred))


seed_everything(CFG.seed)
CFG.out_dir.mkdir(parents=True, exist_ok=True)
CFG.model_dir.mkdir(parents=True, exist_ok=True)
CFG.data_cache_dir.mkdir(parents=True, exist_ok=True)
print(f"Dataset path: {CFG.dataset_path}")
print(f"Working dir : {CFG.out_dir}")

SEED = CFG.seed; np.random.seed(SEED)
NCPU = CFG.n_jobs
 
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 15; DENSE_SPW = 60; DENSE_K = 20; N_SPLITS = 5
 
BEAMS = [
    (10, 20.0, 144.0, 2, "cons"),
    (10,  8.0,  64.0, 2, "loose"),
    ( 8, 35.0, 220.0, 1, "vcons"),
    (10, 14.0,  90.0, 5, "sm5"),
    (20,  4.0,  36.0, 3, "vloose"),
    (12, 12.0, 100.0, 3, "mid"),
    (15, 25.0, 180.0, 2, "stiff"),
]
 
PF_N = 600; ANCC_N = 600
PF_MOM = 0.993; PF_VN = 0.005; PF_PN = 0.01
PF_GR_SIG_MIN = 10.; PF_GR_SIG_MAX = 60.; PF_GR_SIG_DEF = 30.
PF_INIT_V_STD = 0.02; PF_INIT_SPR = 0.5; PF_RESAMP = 0.5
PF_ROUGH_P = 0.2; PF_ROUGH_V = 0.003; PF_GR_WIN = 5; PF_GR_WT = 0.3
ANCC_ALPHA = 0.998; ANCC_RN = 0.002; ANCC_PN = 0.005
ANCC_IR = 0.01; ANCC_IS = 0.3; ANCC_RP = 0.1; ANCC_RR = 0.001
 
DTW_RADII = (20, 50, 100, 200)
DTW_STOCH_K = 12
DTW_STOCH_TEMP = 3.0
 
 
@njit(cache=False)
def _interp1(grid, v, vmin, step):
    i = int((v - vmin) / step)
    if i < 0: return grid[0]
    n = len(grid) - 1
    if i >= n: return grid[n]
    t = (v - vmin) / step - i
    return grid[i] * (1. - t) + grid[i + 1] * t
 
 
@njit(cache=False)
def _resamp(pos, aux, w, N, rp, rv):
    cum = np.zeros(N + 1)
    for j in range(N): cum[j + 1] = cum[j] + w[j]
    u0 = np.random.uniform(0., 1. / N)
    np2 = np.empty(N); na = np.empty(N); ci = 0
    for j in range(N):
        u = u0 + j / N
        while ci < N - 1 and cum[ci + 1] < u: ci += 1
        np2[j] = pos[ci] + rp * np.random.randn()
        na[j] = aux[ci] + rv * np.random.randn()
    return np2, na
 
 
@njit(cache=False)
def _beam_jit(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30);     bcost[0] = 0.; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt: continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni: fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]: cC[fnd] = tot; cP[fnd] = bi
                else:
                    if nc < MAX: cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]: mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]: best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1): path[s] = hI[s, b]; b = hP[s, b]
    return path
 
 
@njit(cache=False)
def _dtw_sakoe_chiba(query, ref, radius):
    """
    Constrained DTW with Sakoe-Chiba band.
    Returns (cost_matrix, accumulated_cost_matrix, path_i, path_j).
    Uses slanted band: diagonal from (0,0) to (N-1,M-1).
    """
    N = len(query); M = len(ref)
    INF = 1e18
    D = np.full((N, M), INF)
 
    slope = (M - 1.0) / max(N - 1.0, 1.0)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            cost = (query[i] - ref[j]) ** 2
            if i == 0 and j == 0:
                D[i, j] = cost
            elif i == 0:
                prev = D[i, j - 1]
                D[i, j] = cost + (prev if prev < INF else INF)
            elif j == 0:
                prev = D[i - 1, j]
                D[i, j] = cost + (prev if prev < INF else INF)
            else:
                a = D[i - 1, j - 1]
                b = D[i - 1, j]
                c = D[i, j - 1]
                mn = a if a < b else b
                mn = mn if mn < c else c
                D[i, j] = cost + (mn if mn < INF else INF)
 
    i = N - 1; j = M - 1
    pi = np.zeros(N + M, np.int64)
    pj = np.zeros(N + M, np.int64)
    k = 0
    while i > 0 or j > 0:
        pi[k] = i; pj[k] = j; k += 1
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
            if a <= b and a <= c:
                i -= 1; j -= 1
            elif b <= c:
                i -= 1
            else:
                j -= 1
    pi[k] = 0; pj[k] = 0; k += 1
    return D, pi[:k], pj[:k]
 
 
@njit(cache=False)
def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
    j_for_i = np.zeros(N, np.int64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = pj[k]
    result = np.empty(N, np.float32)
    for i in range(N):
        result[i] = tw_tvt[j_for_i[i]]
    return result
 
 
@njit(cache=False)
def _dtw_path_slope(pi, pj, N, smooth_win=5):
    j_for_i = np.zeros(N, np.float64)
    for k in range(len(pi)):
        j_for_i[pi[k]] = float(pj[k])
 
    slope = np.zeros(N, np.float32)
    hw = smooth_win // 2
    for i in range(N):
        i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
        if i1 > i0:
            slope[i] = float((j_for_i[i1] - j_for_i[i0]) / (i1 - i0))
        else:
            slope[i] = 1.0
    return slope
 
 
@njit(cache=False)
def _dtw_stochastic_realizations(query, ref, radius, K, temperature):
    N = len(query); M = len(ref)
    INF = 1e18
    slope = (M - 1.0) / max(N - 1.0, 1.0)
 
    D_base = np.full((N, M), INF)
    for i in range(N):
        j_center = int(round(i * slope))
        j_lo = max(0, j_center - radius)
        j_hi = min(M - 1, j_center + radius)
        for j in range(j_lo, j_hi + 1):
            D_base[i, j] = (query[i] - ref[j]) ** 2
 
    paths = np.zeros((K, N), np.int64)
    for k in range(K):
        D = np.full((N, M), INF)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                noise = -temperature * np.log(-np.log(np.random.uniform(1e-10, 1.0)))
                cost = D_base[i, j] + noise
                if i == 0 and j == 0:
                    D[i, j] = cost
                elif i == 0:
                    prev = D[i, j - 1]
                    D[i, j] = cost + (prev if prev < INF else INF)
                elif j == 0:
                    prev = D[i - 1, j]
                    D[i, j] = cost + (prev if prev < INF else INF)
                else:
                    a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                    mn = a if a < b else b
                    mn = mn if mn < c else c
                    D[i, j] = cost + (mn if mn < INF else INF)
 
        i = N - 1; j = M - 1
        j_for_i = np.zeros(N, np.int64)
        while i > 0 or j > 0:
            j_for_i[i] = j
            if i == 0:
                j -= 1
            elif j == 0:
                i -= 1
            else:
                a = D[i - 1, j - 1]; b = D[i - 1, j]; c = D[i, j - 1]
                if a <= b and a <= c:
                    i -= 1; j -= 1
                elif b <= c:
                    i -= 1
                else:
                    j -= 1
        j_for_i[0] = j
        paths[k] = j_for_i
 
    return paths
 
 
@njit(cache=False)
def _pf_ancc(md_v, z_v, gr_v, gg, vmin, step, gs, ls, ir, N,
             ALPHA, RN, PN, IS, RP, RR, RESAMP):
    pos = np.empty(N); rate = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ls + IS * np.random.randn()
        rate[j] = ir + 0.01 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        for j in range(N):
            rate[j] = ALPHA * rate[j] + RN * np.random.randn()
            pos[j] += rate[j] * dm + PN * np.random.randn()
            tvt_j = pos[j] - z_v[i]
            tvt_j = max(tvt_j, vmin - 50.); tvt_j = min(tvt_j, vmin + len(gg) * step + 50.)
            pos[j] = tvt_j + z_v[i]
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                eg = _interp1(gg, pos[j] - z_v[i], vmin, step)
                d = (gr_v[i] - eg) / gs
                lk = max(np.exp(-0.5 * d * d) if d * d < 600. else 0., 1e-300)
                w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, rate = _resamp(pos, rate, w, N, RP, RR)
            for j in range(N): w[j] = 1. / N
        tv = 0.
        for j in range(N): tv += w[j] * (pos[j] - z_v[i])
        pts[i] = tv; va = 0.
        for j in range(N): va += w[j] * (pos[j] - z_v[i] - tv) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]
    return pts, std_
 
 
@njit(cache=False)
def _pf_z(md_v, z_v, gr_v, gr_sm_v, gg_p, gg_s, vmin, step,
          gs, ip, iv, beta, icpt, zsig, N,
          MOM, VN, PN, GR_WT, RP, RV, RESAMP):
    pos = np.empty(N); vel = np.empty(N); w = np.ones(N) / N
    for j in range(N):
        pos[j] = ip + 0.5 * np.random.randn()
        vel[j] = iv + 0.02 * np.random.randn()
    pts = np.empty(len(md_v)); std_ = np.empty(len(md_v)); pm = md_v[0] - 1.; pz = z_v[0] - 1.
    for i in range(len(md_v)):
        dm = md_v[i] - pm; dm = max(dm, 1.)
        dzd = (z_v[i] - pz) / dm; ve = beta * dzd + icpt
        for j in range(N):
            vel[j] = MOM * vel[j] + VN * np.random.randn()
            pos[j] += vel[j] * dm + PN * np.random.randn()
            pos[j] = max(pos[j], vmin - 50.); pos[j] = min(pos[j], vmin + len(gg_p) * step + 50.)
        if not np.isnan(gr_v[i]):
            ws = 0.
            for j in range(N):
                ep = _interp1(gg_p, pos[j], vmin, step)
                dp = (gr_v[i] - ep) / gs
                lp = max(np.exp(-0.5 * dp * dp) if dp * dp < 600. else 0., 1e-300)
                if not np.isnan(gr_sm_v[i]):
                    es = _interp1(gg_s, pos[j], vmin, step)
                    ds = (gr_sm_v[i] - es) / (gs * 1.5)
                    ls = max(np.exp(-0.5 * ds * ds) if ds * ds < 600. else 0., 1e-300)
                    lk = (1. - GR_WT) * lp + GR_WT * ls
                else:
                    lk = lp
                lk = max(lk, 1e-300); w[j] *= lk; ws += w[j]
            if ws > 0.:
                for j in range(N): w[j] /= ws
            else:
                for j in range(N): w[j] = 1. / N
        ws2 = 0.
        for j in range(N):
            dv = (vel[j] - ve) / max(zsig * 2., 0.005)
            lz = max(np.exp(-0.5 * dv * dv) if dv * dv < 600. else 0., 1e-300)
            w[j] *= lz; ws2 += w[j]
        if ws2 > 0.:
            for j in range(N): w[j] /= ws2
        else:
            for j in range(N): w[j] = 1. / N
        ne = 0.
        for j in range(N): ne += w[j] * w[j]
        if 1. / ne < RESAMP * N:
            pos, vel = _resamp(pos, vel, w, N, RP, RV)
            for j in range(N): w[j] = 1. / N
        wm = 0.
        for j in range(N): wm += w[j] * pos[j]
        pts[i] = wm; va = 0.
        for j in range(N): va += w[j] * (pos[j] - wm) ** 2
        std_[i] = va ** 0.5; pm = md_v[i]; pz = z_v[i]
    return pts, std_
 
 
def _grid(tw_tvt, tw_gr, step=0.2):
    tmin = float(tw_tvt.min()); tmax = float(tw_tvt.max())
    tvt_g = np.arange(tmin, tmax + step, step)
    return np.interp(tvt_g, tw_tvt, tw_gr).astype(np.float64), float(tmin), float(step)
 
 
def _gr_sig(hw, tw_tvt, tw_gr):
    kn = hw[hw['TVT_input'].notna() & hw['GR'].notna()]
    if len(kn) < 20: return float(PF_GR_SIG_DEF)
    return float(np.clip(np.std(kn['GR'].values - np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)),
                         PF_GR_SIG_MIN, PF_GR_SIG_MAX))
 
 
def _nn(arr, v):
    i = int(np.searchsorted(arr, v, 'left'))
    if i >= len(arr): return len(arr) - 1
    if i > 0 and abs(arr[i - 1] - v) <= abs(arr[i] - v): return i - 1
    return i
 
 
def _smooth(vals, fb, r):
    s = pd.Series(vals, dtype='float32').interpolate(limit_direction='both').fillna(fb)
    return (s.rolling(r * 2 + 1, center=True, min_periods=1).mean() if r > 0 else s).to_numpy(np.float32)

def geology_boundary_distance(query_tvt: np.ndarray, tw: pd.DataFrame, fill_value: float = 999.0) -> np.ndarray:
    """
    Distance from query TVT values to the nearest Geology boundary in the typewell.
    Returns float32 array with the same length as query_tvt.
    """
    query_tvt = np.asarray(query_tvt, dtype=np.float32)

    if "Geology" not in tw.columns or "TVT" not in tw.columns or len(tw) < 2:
        return np.full(len(query_tvt), fill_value, dtype=np.float32)

    tmp = tw[["TVT", "Geology"]].dropna().sort_values("TVT")
    if len(tmp) < 2:
        return np.full(len(query_tvt), fill_value, dtype=np.float32)

    tvt = tmp["TVT"].to_numpy(np.float32)
    geo = tmp["Geology"].astype(str).to_numpy()

    change_idx = np.where(geo[1:] != geo[:-1])[0]
    if len(change_idx) == 0:
        return np.full(len(query_tvt), fill_value, dtype=np.float32)

    boundaries = ((tvt[change_idx] + tvt[change_idx + 1]) / 2.0).astype(np.float32)

    idx = np.searchsorted(boundaries, query_tvt)

    left_dist = np.full(len(query_tvt), fill_value, dtype=np.float32)
    right_dist = np.full(len(query_tvt), fill_value, dtype=np.float32)

    m_left = idx > 0
    left_dist[m_left] = np.abs(query_tvt[m_left] - boundaries[idx[m_left] - 1])

    m_right = idx < len(boundaries)
    right_dist[m_right] = np.abs(query_tvt[m_right] - boundaries[idx[m_right]])

    dist = np.minimum(left_dist, right_dist)
    return np.clip(dist, 0.0, fill_value).astype(np.float32)
 
 
def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs, mc, es, r):
    si = _nn(tw_tvt, start_tvt)
    sgr = _smooth(gr_h, float(np.nanmean(tw_gr)), r).astype(np.float64)
    path = _beam_jit(sgr, tw_gr.astype(np.float64), si, bs, float(mc), float(es))
    return tw_tvt[path].astype(np.float32)
 
 
def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    ls = float(kn['TVT_input'].iloc[-1] + kn['Z'].iloc[-1])
    tail = kn.tail(30); dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values); dm = np.diff(tail['MD'].values); m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    pts, std = _pf_ancc(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                        ev['GR'].values.astype(np.float64), gg, gmin, gst,
                        gs, ls, ir, N, ANCC_ALPHA, ANCC_RN, ANCC_PN, ANCC_IS, ANCC_RP, ANCC_RR, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)
 
 
def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
    gs = _gr_sig(hw, tw_tvt, tw_gr)
    tw_s = pd.Series(tw_gr).rolling(PF_GR_WIN, center=True, min_periods=1).mean().values.astype(np.float32)
    kna = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0: return np.array([]), np.array([])
    dz_k = np.diff(kna['Z'].values); dvt = np.diff(kna['TVT_input'].values)
    dmd_k = np.diff(kna['MD'].values); m2 = dmd_k > 0
    if m2.sum() >= 10:
        vz = dz_k[m2] / dmd_k[m2]; vt = dvt[m2] / dmd_k[m2]
        A = np.column_stack([vz, np.ones_like(vz)]); c, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
        beta, icpt, zsig = float(c[0]), float(c[1]), max(float(np.std(vt - (c[0] * vz + c[1]))), 0.001)
    else:
        beta, icpt, zsig = -1., 0., 0.1
    t2 = kna.tail(20); dvt2 = np.diff(t2['TVT_input'].values); dmd2 = np.diff(t2['MD'].values); m3 = dmd2 > 0
    iv = float(np.median(dvt2[m3] / dmd2[m3])) if m3.sum() >= 3 else 0.
    gg, gmin, gst = _grid(tw_tvt, tw_gr)
    gs2, _, _ = _grid(tw_tvt, tw_s)
    gr_sm = hw['GR'].rolling(PF_GR_WIN, center=True, min_periods=1).mean()
    pts, std = _pf_z(ev['MD'].values.astype(np.float64), ev['Z'].values.astype(np.float64),
                     ev['GR'].values.astype(np.float64),
                     gr_sm.loc[ev.index].values.astype(np.float64),
                     gg, gs2, gmin, gst, gs, float(kna['TVT_input'].iloc[-1]), iv,
                     beta, icpt, zsig, N,
                     PF_MOM, PF_VN, PF_PN, PF_GR_WT, PF_ROUGH_P, PF_ROUGH_V, PF_RESAMP)
    return pts.astype(np.float32), std.astype(np.float32)
 
 
def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
    N = len(query_gr)
    qn = (query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)
    rn = (tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)
 
    si = _nn(tw_tvt, last_tvt)
    qn_f = qn.astype(np.float64)
    rn_f = rn.astype(np.float64)
 
    dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
    inv_cost_sum = 0.0
    tvt_stack = []
 
    for r in radii:
        D, pi, pj = _dtw_sakoe_chiba(qn_f, rn_f, r)
        cost = float(D[len(qn_f) - 1, len(rn_f) - 1]) / max(len(qn_f) + len(rn_f), 1)
        tvt_pred = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt.astype(np.float32), N)
        slope = _dtw_path_slope(pi[::-1], pj[::-1], N)
        dtw_tvts[r] = tvt_pred
        dtw_slopes[r] = slope
        dtw_costs[r] = cost
        ic = 1.0 / (cost + 1e-6)
        inv_cost_sum += ic
        tvt_stack.append((tvt_pred, ic))
 
    weights = np.array([ic / inv_cost_sum for _, ic in tvt_stack], dtype=np.float32)
    tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
    dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
 
    return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens
 
 
def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt,
                       radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP):
    N = len(query_gr)
    qn = ((query_gr - query_gr.mean()) / (query_gr.std() + 1e-6)).astype(np.float64)
    rn = ((tw_gr - tw_gr.mean()) / (tw_gr.std() + 1e-6)).astype(np.float64)
 
    paths = _dtw_stochastic_realizations(qn, rn, radius, K, temperature)
    tvt_realiz = np.empty((K, N), dtype=np.float32)
    for k in range(K):
        for i in range(N):
            tvt_realiz[k, i] = tw_tvt[paths[k, i]]
 
    mean_tvt = tvt_realiz.mean(axis=0).astype(np.float32)
    std_tvt = tvt_realiz.std(axis=0).astype(np.float32)
    cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
    return mean_tvt, std_tvt, cv_tvt
 
 
_md = np.linspace(1, 50, 20, np.float64); _z = np.zeros(20, np.float64); _gr = np.full(20, 50., np.float64)
_gg = np.linspace(45, 55, 100, np.float64)
_q = np.random.randn(40); _r = np.random.randn(50)
 
def robust_slope(x, y, w=None):
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 2 or np.std(x[m]) < 1e-6: return 0.
    return float(np.polyfit(x[m], y[m], 1)[0])
 
 
def affine_cal(kgr, tw_at_k, min_pts=20):
    v = np.isfinite(kgr) & np.isfinite(tw_at_k)
    if v.sum() < min_pts or np.std(tw_at_k[v]) < 1e-6:
        return 1., float(np.nanmean(kgr) - np.nanmean(tw_at_k)) if v.any() else 0.
    a, b = np.polyfit(tw_at_k[v], kgr[v], 1); return float(a), float(b)
 
 
def seg_b_well(ktvt, kz, form_col):
    bv = ktvt + kz - form_col; n = len(bv)
    b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n - 50):])) if n >= 5 else b_full
    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02 * np.arange(n)); w /= w.sum()
    b_wls = float(np.dot(w, bv))
    return b_full, b_early, b_mid, b_late, b_wls
 
 
def multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win; best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    tvts = np.stack([o[0] for o in out], 1); scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3. * scores); sw /= sw.sum(1, keepdims=True) + 1e-9
    sc_ens = (tvts * sw).sum(1).astype(np.float32)
    return out, sc_ens
 
 
class FormationPlaneKNN:
    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try: df = pd.read_csv(p, usecols=['X', 'Y'] + FORMATIONS).dropna()
            except: continue
            if len(df) == 0: continue
            row = {'wid': wid, 'x': float(df['X'].median()), 'y': float(df['Y'].median())}
            for c in FORMATIONS: row[f'{c}_m'] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows); self.wmap = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy(); self.scale = np.where(xy.std(0) < 1e-3, 1., xy.std(0))
        self.tree = cKDTree(xy / self.scale)
        self.xa = self.df['x'].to_numpy(); self.ya = self.df['y'].to_numpy()
        self.fa = self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)
 
    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale; nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid in self.wmap: dist = np.where(idx == self.wmap[self_wid], np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.).astype(np.float64)
        xn = self.xa[ik]; yn = self.ya[ik]; fn = self.fa[ik]; wx = w * xn; wy = w * yn
        A = np.zeros((len(q), 3, 3))
        A[:, 0, 0] = (wx * xn).sum(1); A[:, 0, 1] = (wx * yn).sum(1); A[:, 0, 2] = wx.sum(1)
        A[:, 1, 0] = A[:, 0, 1]; A[:, 1, 1] = (wy * yn).sum(1); A[:, 1, 2] = wy.sum(1)
        A[:, 2, 0] = A[:, 0, 2]; A[:, 2, 1] = A[:, 1, 2]; A[:, 2, 2] = w.sum(1)
        A[:, 0, 0] += 1e-9; A[:, 1, 1] += 1e-9; A[:, 2, 2] += 1e-9
        rhs = np.stack([(wx[:, :, None] * fn).sum(1), (wy[:, :, None] * fn).sum(1),
                        (w[:, :, None] * fn).sum(1)], 1)
        try:
            coef = np.linalg.solve(A, rhs)
        except:
            coef = np.zeros((len(q), 3, 6))
            for r in range(len(q)):
                try: coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except: pass
        Xq = xy_q[:, 0]; Yq = xy_q[:, 1]
        pred = (Xq[:, None] * coef[:, 0, :] + Yq[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)
        pred[~vk.any(1)] = self.fa.mean(0)
        return pred, np.where(vk, dk, np.inf).min(1).astype(np.float32)
 
 
class DenseANCCImputer:
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try: df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC']).dropna()
            except: continue
            if len(df) == 0: continue
            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int); s = df.iloc[ix]
            xs.append(s['X'].values); ys.append(s['Y'].values)
            anccs.append(s['ANCC'].values); wids.extend([wid] * len(s))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32); self.wids = np.array(wids)
        self.scale = np.where(self.xy.std(0) < 1e-3, 1., self.xy.std(0))
        self.tree = cKDTree(self.xy / self.scale)
 
    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(xy_q); q = xy_q / self.scale; nf = min(nfetch, len(self.ancc))
        dist, idx = self.tree.query(q, k=nf, workers=-1)
        if self_wid: dist = np.where(self.wids[idx] == self_wid, np.inf, dist)
        ord = np.argpartition(dist, min(k - 1, nf - 1), 1)[:, :k]
        dk = np.take_along_axis(dist, ord, 1); ik = np.take_along_axis(idx, ord, 1)
        vk = np.isfinite(dk); w = np.where(vk, 1. / (dk + 1e-3), 0.)
        sw = w.sum(1); safe = np.where(sw < 1e-9, 1., sw); an = self.ancc[ik]
        ap = (an * w).sum(1) / safe; ap = np.where(sw < 1e-9, float(self.ancc.mean()), ap)
        var = ((an - ap[:, None]) ** 2 * w).sum(1) / safe
        return ap.astype(np.float32), np.sqrt(np.maximum(var, 0.)).astype(np.float32), \
               np.where(vk, dk, np.inf).min(1).astype(np.float32)
 
 
_FI = None
_DI = None


def init_spatial_imputers():
    global _FI, _DI
    if _FI is None or _DI is None:
        hw_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
        train_wids = [p.stem.replace('__horizontal_well', '') for p in hw_paths]
        _FI = FormationPlaneKNN(train_wids, CFG.dataset_path / "train")
        _DI = DenseANCCImputer(train_wids, CFG.dataset_path / "train")
    return _FI, _DI
ANCH_OFFS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], np.float32)
BEAM_OFFS = np.array([-40, -20, -10, -5, -3, 0, 3, 5, 10, 20, 40], np.float32)
SC_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
PF_OFFS   = np.array([-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30], np.float32)
DTW_OFFS  = np.array([-20, -10, -5, -2, 0, 2, 5, 10, 20], np.float32)
 
 
def build_well(hw_path, tw_path, is_train):
    global _FI, _DI
    wid = Path(hw_path).stem.replace('__horizontal_well', '')
    try:
        hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path).sort_values('TVT')
    except:
        return None
    if is_train and 'TVT' not in hw.columns: return None
    kn = hw[hw['TVT_input'].notna()]; ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0 or len(kn) < 10: return None
    if is_train and hw['TVT'].isna().all(): return None
    tw_tvt = tw['TVT'].to_numpy(np.float32); tw_gr = tw['GR'].to_numpy(np.float32)
    if len(tw_tvt) < 3: return None
 
    pf_a, std_a = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_a) == 0: return None
    pf_z, std_z = run_pf_z(hw, tw_tvt, tw_gr)
    pf_use = pf_a.astype(np.float32); std_use = std_a.astype(np.float32)
    has_z = len(pf_z) == len(pf_a) and not np.any(np.isnan(pf_z))
    geo_boundary_dist_pf = geology_boundary_distance(pf_use, tw)
 
    lk = kn.iloc[-1]; last_tvt = float(lk['TVT_input'])
    gr_full = hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
    hgr = gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
    kgr = gr_full.iloc[:len(kn)].to_numpy(np.float32)
 
    bpaths = {}
    for (bs, mc, es, r, tag) in BEAMS:
        bpaths[tag] = beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
    beam_ref = (bpaths['cons'] + bpaths['sm5']) / 2.
 
    ktvt = kn['TVT_input'].to_numpy(np.float32)
    sc_res, sc_ens = multi_scale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3)
    sc8, sc8s = sc_res[0]; sc15, sc15s = sc_res[1]; sc25, sc25s = sc_res[2]
    sc_cons = (sc8 + sc15 + sc25) / 3.
    sc_trust = float(np.clip(len(kn) / 200., 0., 0.6))
    hyb_ref = (1 - sc_trust) * beam_ref + sc_trust * sc_ens
 
    full_gr = gr_full.values.astype(np.float32)
    dtw_tvts_ms, dtw_slopes_ms, dtw_costs_ms, dtw_ens_ms = run_dtw_multiscale(
        full_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII
    )
 
    dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
        full_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP
    )
 
    nh = len(ev)
    ev_start = ev.index[0]
 
    def _ev_slice(arr):
        return arr[ev_start:ev_start + nh].astype(np.float32)
 
    dtw_ens_ev = _ev_slice(dtw_ens_ms)
    dtw_mean_ev = _ev_slice(dtw_mean_stoch)
    dtw_std_ev = _ev_slice(dtw_std_stoch)
    dtw_cv_ev = _ev_slice(dtw_cv_stoch)
 
    dtw_per_radius_ev = {}
    dtw_slope_ev = {}
    for r in DTW_RADII:
        dtw_per_radius_ev[r] = _ev_slice(dtw_tvts_ms[r])
        dtw_slope_ev[r] = _ev_slice(dtw_slopes_ms[r])
 
    dtw_slope_mean_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
    dtw_ens_slope_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
 
    dtw_cost_arr = np.array([dtw_costs_ms[r] for r in DTW_RADII], dtype=np.float32)
    dtw_cost_min = float(dtw_cost_arr.min())
    dtw_cost_range = float(dtw_cost_arr.max() - dtw_cost_arr.min())
 
    tw_at_k = np.interp(ktvt, tw_tvt, tw_gr).astype(np.float32)
    a_cal, b_cal = affine_cal(kgr, tw_at_k)
    kmd = kn['MD'].to_numpy(np.float32); kz = kn['Z'].to_numpy(np.float32)
    pfx_rmse = float(np.sqrt(np.mean((kgr - tw_at_k) ** 2)))
    slp_all = robust_slope(kmd, ktvt); slp_50 = robust_slope(kmd[-50:], ktvt[-50:])
    slp_z = robust_slope(kz, ktvt)
 
    swid = wid if is_train else None
    xy_ev = ev[['X', 'Y']].to_numpy(np.float64); xy_kn = kn[['X', 'Y']].to_numpy(np.float64)
    form_ev, knn_d = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _ = _FI.impute(xy_kn, self_wid=swid)
    z_kn = kn['Z'].to_numpy(np.float32); z_ev = ev['Z'].to_numpy(np.float32)
 
    tvt_fs = {}; form_rmse = {}; form_list = []
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, fi2])
        tvt_f = (-z_ev + form_ev[:, fi2] + b_full).astype(np.float32)
        tvt_fw = (-z_ev + form_ev[:, fi2] + b_wls).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'tvtF_{fn}'] = tvt_f; tvt_fs[f'tvtFw_{fn}'] = tvt_fw
        tvt_fs[f'tvtF50_{fn}'] = tvt_f50
        tvt_fs[f'bw_{fn}'] = np.float32(b_full); tvt_fs[f'bww_{fn}'] = np.float32(b_wls)
        tvt_fs[f'bw50_{fn}'] = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}'] = np.float32(b_early)
        tvt_fs[f'bw_mid_{fn}'] = np.float32(b_mid)
        form_rmse[fn] = float(np.sqrt(np.mean((ktvt - (-z_kn + form_kn[:, fi2] + b_full)) ** 2)))
        form_list.append(tvt_f)
 
    fs = np.stack(form_list, 1)
    form_mean_d = (fs.mean(1) - last_tvt).astype(np.float32)
    form_std_d = fs.std(1).astype(np.float32)
    form_rng_d = (fs.max(1) - fs.min(1)).astype(np.float32)
 
    d_ancc, d_std, d_dist = _DI.impute(xy_ev, self_wid=swid)
    d_kn, d_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)
    b_vd = ktvt + z_kn - d_kn
    _, b_de, b_dm, b_dl, b_dw = seg_b_well(ktvt, z_kn, d_kn)
    b_d = float(np.median(b_vd))
    tvt_dense = (-z_ev + d_ancc + b_d).astype(np.float32)
    tvt_densew = (-z_ev + d_ancc + b_dw).astype(np.float32)
    tvt_dense50 = (-z_ev + d_ancc + b_dl).astype(np.float32)
    res_kn = ktvt + z_kn - d_kn
    d_rmse = float(np.sqrt(np.mean(res_kn ** 2)))
    d_bias = float(np.mean(res_kn)); d_nb_std = float(np.mean(d_std_kn))
 
    all_sigs = [pf_use] + [p for p in bpaths.values()] + \
               [sc8, sc15, sc25, sc_ens, tvt_fs['tvtF_ANCC'], tvt_dense, dtw_ens_ev]
    sig_mat = np.stack(all_sigs, 1)
    sig_std = sig_mat.std(1).astype(np.float32)
    sig_mean = (sig_mat.mean(1) - last_tvt).astype(np.float32)
 
    gr_s = pd.Series(gr_full.values); rolls = {}
    for w in [5, 21, 51, 101]:
        r = gr_s.rolling(w, center=True, min_periods=1)
        rolls[f'grm{w}'] = r.mean().iloc[ev.index].values.astype(np.float32)
        rolls[f'grs{w}'] = r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
    for lag in [1, 5, 15, 30]:
        rolls[f'glag{lag}'] = gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
        rolls[f'glead{lag}'] = gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
    gr_d1 = gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_d2 = gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
    gr_env = gr_s.rolling(21, center=True, min_periods=1).max().iloc[ev.index].values.astype(np.float32)
    gr_nrg = np.sqrt(np.maximum((gr_s ** 2).rolling(21, center=True, min_periods=1).mean(), 0.)
                     ).iloc[ev.index].values.astype(np.float32)
 
    hmd = ev['MD'].to_numpy(np.float32); md_since = hmd - float(lk['MD'])
    slp_b_all = (last_tvt + slp_all * md_since).astype(np.float32)
    slp_b_50 = (last_tvt + slp_50 * md_since).astype(np.float32)
 
    mdd = hw['MD'].diff().replace(0, np.nan)
    dzdmd = (hw['Z'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dxdmd = (hw['X'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
    dydmd = (hw['Y'].diff() / mdd).iloc[ev.index].values.astype(np.float32)
 
    frac = (np.arange(nh) / max(nh - 1, 1)).astype(np.float32)
 
    def sc(v): return np.full(nh, np.float32(v), np.float32)
 
    feats = {
        'well': wid, 'id': [f'{wid}_{i}' for i in ev.index],
        'last_known_tvt': sc(last_tvt),
        'pf_ancc': pf_use, 'pf_ancc_std': std_use,
        'pf_ancc_delta': (pf_use - last_tvt).astype(np.float32),
        'geo_boundary_dist_pf': geo_boundary_dist_pf,
        'pf_z': (pf_z.astype(np.float32) if has_z else sc(last_tvt)),
        'pf_z_delta': ((pf_z - last_tvt).astype(np.float32) if has_z else sc(0.)),
        'pf_vs_z': ((pf_use - pf_z.astype(np.float32)) if has_z else sc(0.)),
        **{f'beam_{t}_d': (p - np.float32(last_tvt)).astype(np.float32) for t, p in bpaths.items()},
        'beam_mean_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).mean(1).astype(np.float32),
        'beam_std_d': np.stack([(p - last_tvt) for p in bpaths.values()], 1).std(1).astype(np.float32),
        'beam_med_d': np.median(np.stack([(p - last_tvt) for p in bpaths.values()], 1), 1).astype(np.float32),
        'sc8_d': (sc8 - np.float32(last_tvt)).astype(np.float32), 'sc8_sc': sc8s,
        'sc15_d': (sc15 - np.float32(last_tvt)).astype(np.float32), 'sc15_sc': sc15s,
        'sc25_d': (sc25 - np.float32(last_tvt)).astype(np.float32), 'sc25_sc': sc25s,
        'sc_cons_d': (sc_cons - np.float32(last_tvt)).astype(np.float32),
        'sc_ens_d': (sc_ens - np.float32(last_tvt)).astype(np.float32),
        'sc_trust': sc(sc_trust), 'hyb_d': (hyb_ref - np.float32(last_tvt)).astype(np.float32),
        'sig_std': sig_std, 'sig_mean_d': sig_mean,
        **tvt_fs,
        **{f'frm_rmse_{fn}': sc(form_rmse[fn]) for fn in FORMATIONS},
        'form_mean_d': form_mean_d, 'form_std_d': form_std_d, 'form_rng_d': form_rng_d,
        'spatial_ancc_d': (form_ev[:, 0] - np.float32(np.interp(last_tvt, tw_tvt, tw_gr))),
        'spatial_knn_dist': knn_d,
        'dense_ancc': d_ancc, 'dense_std': d_std, 'dense_dist': d_dist,
        'tvt_dense_d': (tvt_dense - last_tvt).astype(np.float32),
        'tvt_densew_d': (tvt_densew - last_tvt).astype(np.float32),
        'tvt_dense50_d': (tvt_dense50 - last_tvt).astype(np.float32),
        'dense_rmse': sc(d_rmse), 'dense_bias': sc(d_bias), 'dense_nb_std': sc(d_nb_std),
        'pf_vs_spatial': (pf_use - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'pf_vs_dense': (pf_use - tvt_dense).astype(np.float32),
        'spatial_vs_dense': (tvt_fs['tvtF_ANCC'] - tvt_dense).astype(np.float32),
        'beam_vs_spatial': (bpaths['cons'] - tvt_fs['tvtF_ANCC']).astype(np.float32),
        'sc_vs_beam': (sc_ens - bpaths['cons']).astype(np.float32),
        'dtw_ens_d': (dtw_ens_ev - last_tvt).astype(np.float32),
        'dtw_stoch_mean_d': (dtw_mean_ev - last_tvt).astype(np.float32),
        'dtw_stoch_std': dtw_std_ev,
        'dtw_stoch_cv': dtw_cv_ev,
        'dtw_slope_mean': dtw_slope_mean_ev,
        **{f'dtw_r{r}_d': (dtw_per_radius_ev[r] - last_tvt).astype(np.float32) for r in DTW_RADII},
        **{f'dtw_slope_r{r}': dtw_slope_ev[r] for r in DTW_RADII},
        'dtw_cost_min': sc(dtw_cost_min),
        'dtw_cost_range': sc(dtw_cost_range),
        'dtw_vs_beam': (dtw_ens_ev - bpaths['cons']).astype(np.float32),
        'dtw_vs_pf': (dtw_ens_ev - pf_use).astype(np.float32),
        'dtw_vs_sc': (dtw_ens_ev - sc_ens).astype(np.float32),
        **{f'tddtw{int(o)}': hgr - np.interp(dtw_ens_ev + o, tw_tvt, tw_gr).astype(np.float32)
           for o in DTW_OFFS},
        'cal_a': sc(a_cal), 'cal_b': sc(b_cal),
        'pfx_rmse': sc(pfx_rmse), 'known_len': sc(len(kn)), 'eval_len': sc(nh),
        'slp_all': sc(slp_all), 'slp_50': sc(slp_50), 'slp_z': sc(slp_z),
        'slp_b_d_all': (slp_b_all - last_tvt).astype(np.float32),
        'slp_b_d_50': (slp_b_50 - last_tvt).astype(np.float32),
        'ktvt_range': sc(float(np.ptp(ktvt))), 'ktvt_std': sc(float(ktvt.std())),
        'md_since': md_since, 'frac': frac, 'frac2': frac ** 2, 'sqrt_frac': np.sqrt(frac),
        'z': z_ev,
        'dx': (ev['X'] - float(lk['X'])).to_numpy(np.float32),
        'dy': (ev['Y'] - float(lk['Y'])).to_numpy(np.float32),
        'dz': (z_ev - float(lk['Z'])).astype(np.float32),
        'dxy': np.sqrt((ev['X'] - float(lk['X'])) ** 2 + (ev['Y'] - float(lk['Y'])) ** 2).to_numpy(np.float32),
        'dzdmd': dzdmd, 'dxdmd': dxdmd, 'dydmd': dydmd,
        'gr': hgr, 'gr_d1': gr_d1, 'gr_d2': gr_d2, 'gr_env': gr_env, 'gr_nrg': gr_nrg,
        'gr_vs_tw_anc': hgr - np.float32(np.interp(last_tvt, tw_tvt, tw_gr)),
        'gr_vs_slp_all': hgr - np.interp(slp_b_all, tw_tvt, tw_gr).astype(np.float32),
        **{f'tda{int(o)}': hgr - np.float32(np.interp(last_tvt + o, tw_tvt, tw_gr)) for o in ANCH_OFFS},
        **{f'tdbc{int(o)}': hgr - np.interp(beam_ref + o, tw_tvt, tw_gr).astype(np.float32) for o in BEAM_OFFS},
        **{f'tdsc{int(o)}': hgr - np.interp(sc_ens + o, tw_tvt, tw_gr).astype(np.float32) for o in SC_OFFS},
        **{f'tdpf{int(o)}': hgr - np.interp(pf_use + o, tw_tvt, tw_gr).astype(np.float32) for o in PF_OFFS},
        'tw_range': sc(float(np.ptp(tw_tvt))), 'tw_gr_mean': sc(float(tw_gr.mean())),
    }
    for k, v in rolls.items(): feats[k] = v
    result = pd.DataFrame(feats)
    if is_train:
        if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
        result['target'] = (ev['TVT'].to_numpy(np.float32) - np.float32(last_tvt))
    return result
 
 
def build_dataset(paths, is_train, label):
    init_spatial_imputers()
    args = [(str(p), str(p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv'), is_train)
            for p in paths
            if (p.parent / f'{p.stem.replace("__horizontal_well", "")}__typewell.csv').exists()]
    res = Parallel(n_jobs=NCPU, prefer='threads', verbose=0)(
        delayed(build_well)(hp, tp, it) for hp, tp, it in args)
    parts = [r for r in res if r is not None]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

def make_lgb_params(seed: int) -> list[dict]:
    """Starter params. Replace with your tuned hidden params if available."""
    base = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "seed": seed,
        "feature_fraction_seed": seed,
        "bagging_seed": seed,
        "data_random_seed": seed,
        "num_threads": CFG.n_jobs,
        "force_col_wise": True,
    }
    return [
        dict(
            base,
            n_estimators=10000,
            learning_rate=0.015,
            num_leaves=127,
            min_data_in_leaf=50,
            feature_fraction=0.65,
            bagging_fraction=0.75,
            bagging_freq=1,
            lambda_l1=0.0,
            lambda_l2=10.0,
            max_depth=10,
            extra_trees=True,
        ),
        dict(base, n_estimators=10000, learning_rate=0.014, num_leaves=95,
             min_data_in_leaf=18, feature_fraction=0.72, bagging_fraction=0.80,
             bagging_freq=1, lambda_l1=0.0, lambda_l2=2.0, max_depth=-1),
        #dict(base, n_estimators=8000, learning_rate=0.022, num_leaves=47,
        #     min_data_in_leaf=35, feature_fraction=0.90, bagging_fraction=0.90,
        #     bagging_freq=1, lambda_l1=0.10, lambda_l2=0.5, max_depth=10),
    ]


def make_cb_params(seed: int) -> list[dict]:
    """Starter params. Replace with your tuned hidden params if available."""
    base = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_seed": seed,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": CFG.n_jobs,
        "od_type": "Iter",
        "od_wait": 50,
    }
    return [
        dict(base, iterations=9000, learning_rate=0.025, depth=8, l2_leaf_reg=6.0,
             random_strength=0.7, bagging_temperature=0.4),
        dict(base, iterations=8000, learning_rate=0.022, depth=8,
            l2_leaf_reg=9.0, random_strength=1.2,bagging_temperature=0.8,),
        dict(base, random_seed=seed + 202, iterations=8500, learning_rate=0.020,
            depth=8, l2_leaf_reg=12.0, random_strength=1.8, bagging_temperature=1.2,),
    ]


def load_or_build_features() -> tuple[pd.DataFrame, pd.DataFrame]:
    train_cache = CFG.data_cache_dir / "train_features.csv"
    test_cache = CFG.data_cache_dir / "test_features.csv"

    artifact_train = CFG.artifacts_path / "data" / "train.csv"
    if CFG.use_artifact_features and not CFG.force_rebuild_features and artifact_train.exists():
        print(f"Loading train features from artifact: {artifact_train}")
        train_df = pd.read_csv(artifact_train, low_memory=False)
    elif not CFG.force_rebuild_features and train_cache.exists():
        print(f"Loading train features from cache: {train_cache}")
        train_df = pd.read_csv(train_cache, low_memory=False)
    else:
        hw_paths = sorted((CFG.dataset_path / "train").glob("*__horizontal_well.csv"))
        print(f"Building train features for {len(hw_paths)} wells")
        train_df = build_dataset(hw_paths, is_train=True, label="train")
        train_df.to_csv(train_cache, index=False)

    if not CFG.force_rebuild_features and test_cache.exists():
        print(f"Loading test features from cache: {test_cache}")
        test_df = pd.read_csv(test_cache, low_memory=False)
    else:
        test_paths = sorted((CFG.dataset_path / "test").glob("*__horizontal_well.csv"))
        print(f"Building test features for {len(test_paths)} wells")
        test_df = build_dataset(test_paths, is_train=False, label="test")
        test_df.to_csv(test_cache, index=False)

    return train_df, test_df


def prepare_matrix(train_df: pd.DataFrame, test_df: pd.DataFrame):
    drop_cols = {"well", "id", "target"}
    features = [c for c in train_df.columns if c not in drop_cols]
    features = [c for c in features if pd.api.types.is_numeric_dtype(train_df[c])]

    for c in features:
        if c not in test_df.columns:
            test_df[c] = np.nan

    X = train_df[features].replace([np.inf, -np.inf], np.nan)
    X_test = test_df[features].replace([np.inf, -np.inf], np.nan)

    all_nan = [c for c in features if X[c].isna().all()]
    if all_nan:
        print(f"Dropping all-NaN features: {len(all_nan)}")
        features = [c for c in features if c not in all_nan]
        X = X[features]
        X_test = X_test[features]

    y = train_df["target"].astype(np.float32)
    groups = train_df["well"].astype(str)
    return X, y, groups, X_test, features

def train_lightgbm(params: dict, name: str, X, y, groups, X_test):
    path = CFG.model_dir / name
    path.mkdir(parents=True, exist_ok=True)
    models_path = path / "models.pkl"
    oof_path = path / "oof_preds.pkl"

    if models_path.exists() and oof_path.exists():
        print(f"Loading {name}")
        models = joblib.load(models_path)
        oof_preds = joblib.load(oof_path)
        test_preds = np.zeros(len(X_test), np.float32)
        fold_scores = []
        for fold_idx, (_, valid_idx) in enumerate(GroupKFold(CFG.n_splits).split(X, y, groups=groups)):
            m = models[fold_idx]
            best_iter = getattr(m, "best_iteration", None)
            test_preds += m.predict(X_test, num_iteration=best_iter).astype(np.float32) / CFG.n_splits
            fold_scores.append(rmse(y.iloc[valid_idx], oof_preds[valid_idx]))
        return oof_preds, test_preds, rmse(y, oof_preds), fold_scores

    p = copy.deepcopy(params)
    n_estimators = int(p.pop("n_estimators"))
    oof_preds = np.zeros(len(X), np.float32)
    test_preds = np.zeros(len(X_test), np.float32)
    models, fold_scores = [], []
    print(f"Training {name}")

    for fold_idx, (train_idx, valid_idx) in enumerate(GroupKFold(CFG.n_splits).split(X, y, groups=groups)):
        d_train = lgb.Dataset(X.iloc[train_idx], label=y.iloc[train_idx])
        d_valid = lgb.Dataset(X.iloc[valid_idx], label=y.iloc[valid_idx], reference=d_train)
        model = lgb.train(
            p,
            d_train,
            valid_sets=[d_valid],
            num_boost_round=n_estimators,
            callbacks=[lgb.early_stopping(CFG.early_stopping_rounds, verbose=False)],
        )
        models.append(model)
        best_iter = getattr(model, "best_iteration", None)
        oof_preds[valid_idx] = model.predict(X.iloc[valid_idx], num_iteration=best_iter).astype(np.float32)
        test_preds += model.predict(X_test, num_iteration=best_iter).astype(np.float32) / CFG.n_splits
        score = rmse(y.iloc[valid_idx], oof_preds[valid_idx])
        fold_scores.append(score)
        print(f"  fold {fold_idx}: RMSE={score:.5f}")
        gc.collect()

    joblib.dump(models, models_path)
    joblib.dump(oof_preds, oof_path)
    return oof_preds, test_preds, rmse(y, oof_preds), fold_scores


def train_catboost(params: dict, name: str, X, y, groups, X_test):
    path = CFG.model_dir / name
    path.mkdir(parents=True, exist_ok=True)
    models_path = path / "models.pkl"
    oof_path = path / "oof_preds.pkl"

    if models_path.exists() and oof_path.exists():
        print(f"Loading {name}")
        models = joblib.load(models_path)
        oof_preds = joblib.load(oof_path)
        test_preds = np.zeros(len(X_test), np.float32)
        fold_scores = []
        for fold_idx, (_, valid_idx) in enumerate(GroupKFold(CFG.n_splits).split(X, y, groups=groups)):
            model = models[fold_idx]
            test_preds += model.predict(X_test).astype(np.float32) / CFG.n_splits
            fold_scores.append(rmse(y.iloc[valid_idx], oof_preds[valid_idx]))
        return oof_preds, test_preds, rmse(y, oof_preds), fold_scores

    p = copy.deepcopy(params)
    oof_preds = np.zeros(len(X), np.float32)
    test_preds = np.zeros(len(X_test), np.float32)
    models, fold_scores = [], []
    print(f"Training {name}")

    for fold_idx, (train_idx, valid_idx) in enumerate(GroupKFold(CFG.n_splits).split(X, y, groups=groups)):
        model = CatBoostRegressor(**p)
        model.fit(
            Pool(X.iloc[train_idx], label=y.iloc[train_idx]),
            eval_set=Pool(X.iloc[valid_idx], label=y.iloc[valid_idx]),
            use_best_model=True,
        )
        models.append(model)
        oof_preds[valid_idx] = model.predict(X.iloc[valid_idx]).astype(np.float32)
        test_preds += model.predict(X_test).astype(np.float32) / CFG.n_splits
        score = rmse(y.iloc[valid_idx], oof_preds[valid_idx])
        fold_scores.append(score)
        print(f"  fold {fold_idx}: RMSE={score:.5f}")
        gc.collect()

    joblib.dump(models, models_path)
    joblib.dump(oof_preds, oof_path)
    return oof_preds, test_preds, rmse(y, oof_preds), fold_scores


class GreedyBlender:

    def __init__(self, precision: float = 0.001, max_weight: float = 0.5,
                 max_iter: int = 25, min_improve: float = 1e-7):
        self.precision = precision
        self.max_weight = max_weight
        self.max_iter = max_iter
        self.min_improve = min_improve
        self.steps: list[tuple[str, float]] = []
        self.best_score: float | None = None

    def fit(self, preds: pd.DataFrame, y: pd.Series):
        scores = {c: rmse(y, preds[c].values) for c in preds.columns}
        best_col = min(scores, key=scores.get)
        ens = preds[best_col].values.astype(np.float32).copy()
        self.steps = [(best_col, 1.0)]
        best_score = scores[best_col]
        print("Greedy blending start")
        print(f"  start: {best_col} RMSE={best_score:.6f}")

        grid = np.arange(self.precision, self.max_weight + self.precision / 2, self.precision)
        for it in range(self.max_iter):
            iter_best = (best_score, None, None, None)
            for col in preds.columns:
                p = preds[col].values.astype(np.float32)
                for w in grid:
                    cand = (1.0 - w) * ens + w * p
                    score = rmse(y, cand)
                    if score < iter_best[0]:
                        iter_best = (score, col, float(w), cand.astype(np.float32))
            score, col, w, cand = iter_best
            improvement = best_score - score
            if col is None or improvement <= self.min_improve:
                break
            ens = cand
            best_score = score
            self.steps.append((col, w))
            print(f"  iter {it + 1}: +{col} w={w:.3f} RMSE={best_score:.6f} improve={improvement:.6f}")

        self.best_score = best_score
        return self

    def predict(self, preds: pd.DataFrame) -> np.ndarray:
        if not self.steps:
            raise RuntimeError("GreedyBlender is not fitted.")
        first_col, _ = self.steps[0]
        ens = preds[first_col].values.astype(np.float32).copy()
        for col, w in self.steps[1:]:
            ens = ((1.0 - w) * ens + w * preds[col].values.astype(np.float32)).astype(np.float32)
        return ens


def apply_pp(df: pd.DataFrame, model_delta: np.ndarray, pf_delta: np.ndarray,
             alpha: float, tau: int, w_pf: float) -> np.ndarray:
    d = model_delta * (1.0 - w_pf) + pf_delta * w_pf
    if tau:
        gate = 1.0 - np.exp(-np.maximum(df["md_since"].values, 0.0) / tau)
        d = d * gate
    return d * alpha


def sg_smooth(df: pd.DataFrame, col: str, sg_w: int = 17, sg_p: int = 3) -> pd.DataFrame:
    out = df.copy()
    for _, g in out.groupby("well", sort=False):
        v = g[col].values.astype(float)
        n = len(v)
        wl = min(sg_w, n)
        if wl % 2 == 0:
            wl -= 1
        if wl >= sg_p + 2:
            v = savgol_filter(v, wl, sg_p)
        out.loc[g.index, col] = v
    return out


def optimize_postprocess(train_df: pd.DataFrame, y: pd.Series, hc_oof_delta: np.ndarray) -> dict:
    base = train_df["last_known_tvt"].values.astype(np.float32)
    ytrue = y.values.astype(np.float32) + base
    pf_oof_delta = train_df["pf_ancc"].values.astype(np.float32) - base

    def objective(trial):
        alpha = trial.suggest_float("alpha", 0.50, 1.05, step=0.01)
        tau = trial.suggest_int("tau", 5, 700, step=5)
        w_pf = trial.suggest_float("w_pf", 0.00, 0.60, step=0.01)
        d = apply_pp(train_df, hc_oof_delta, pf_oof_delta, alpha, tau, w_pf)
        return rmse(ytrue, base + d)

    sampler = optuna.samplers.TPESampler(seed=CFG.seed, n_startup_trials=40)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=CFG.pp_trials, n_jobs=CFG.pp_n_jobs, show_progress_bar=False)
    print(f"Best postprocess RMSE={study.best_value:.6f}, params={study.best_params}")
    return dict(study.best_params, score=float(study.best_value))


def make_submission(train_df: pd.DataFrame, test_df: pd.DataFrame,
                    hc_test_delta: np.ndarray, pp: dict) -> pd.DataFrame:
    test_df2 = test_df.copy()
    base = test_df2["last_known_tvt"].values.astype(np.float32)
    pf_test_delta = test_df2["pf_ancc"].values.astype(np.float32) - base
    test_df2["pred"] = base + apply_pp(
        test_df2,
        hc_test_delta,
        pf_test_delta,
        pp["alpha"],
        pp["tau"],
        pp["w_pf"],
    )
    test_df2 = sg_smooth(test_df2, "pred")

    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub = sample_sub[["id"]].merge(
        test_df2[["id", "pred"]].rename(columns={"pred": "tvt"}),
        on="id",
        how="left",
    )
    fallback = float(train_df["last_known_tvt"].mean() + train_df["target"].mean())
    sub["tvt"] = sub["tvt"].fillna(fallback)
    sub[["id", "tvt"]].to_csv(CFG.out_dir / "submission.csv", index=False)
    return sub

def main() -> None:
    t0 = time.time()
    train_df, test_df = load_or_build_features()
    print(f"train_df={train_df.shape}, test_df={test_df.shape}")

    X, y, groups, X_test, features = prepare_matrix(train_df, test_df)
    print(f"Using {len(features)} features")

    oof_dict: dict[str, np.ndarray] = {}
    test_dict: dict[str, np.ndarray] = {}
    overall_scores: dict[str, float] = {}
    fold_scores: dict[str, list[float]] = {}

    for i, params in enumerate(make_lgb_params(CFG.seed), 1):
        name = f"lightgbm-{i}"
        oof, pred, score, fs = train_lightgbm(params, name, X, y, groups, X_test)
        oof_dict[name] = oof
        test_dict[name] = pred
        overall_scores[name] = score
        fold_scores[name] = fs
        print(f"{name} overall RMSE={score:.6f}\n")

    for i, params in enumerate(make_cb_params(CFG.seed), 1):
        name = f"catboost-{i}"
        oof, pred, score, fs = train_catboost(params, name, X, y, groups, X_test)
        oof_dict[name] = oof
        test_dict[name] = pred
        overall_scores[name] = score
        fold_scores[name] = fs
        print(f"{name} overall RMSE={score:.6f}\n")

    oof_df = pd.DataFrame(oof_dict)
    test_pred_df = pd.DataFrame(test_dict)

    print("Single-model scores:")
    for k, v in sorted(overall_scores.items(), key=lambda kv: kv[1]):
        print(f"  {k:12s} {v:.6f}")

    blender = GreedyBlender(precision=0.001, max_weight=0.5, max_iter=25).fit(oof_df, y)
    hc_oof_delta = blender.predict(oof_df)
    hc_test_delta = blender.predict(test_pred_df)
    print(f"Greedy blend RMSE={blender.best_score:.6f}")
    print(f"Blend steps={blender.steps}")

    pp = optimize_postprocess(train_df, y, hc_oof_delta)
    sub = make_submission(train_df, test_df, hc_test_delta, pp)
    print(sub.head())
    print(f"Wrote: {CFG.out_dir / 'submission.csv'}")
    print(f"Elapsed: {(time.time() - t0) / 60:.2f} min")
    return sub

sub = main()
